# Mistral Document AI on Microsoft Foundry

This notebook demonstrates passwordless Microsoft Entra ID authentication, an optional API-key flow, and a starter request for processing a local document.

Before running it:

1. Copy `.env.example` to `.env`.
2. Confirm the Foundry resource endpoint and OCR deployment name in `.env`.
3. Sign in locally with `az login` and ensure your identity has the **Cognitive Services User** role on the Foundry resource.
4. Add a document to the `documents` folder.

## 1. Run with uv

From a terminal, run `uv sync`, then `uv run jupyter lab`. You do not need to activate the `.venv`; `uv run` selects it automatically.

In [1]:
import base64
import json
import mimetypes
import os
from pathlib import Path
from typing import Any

import requests
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

True

## 2. Configure the Foundry connection

`AZURE_FOUNDRY_ENDPOINT` identifies the Foundry resource. The Mistral provider route below is the callable OCR endpoint shown for the `mistral-document-ai-2512` deployment.

In [3]:
FOUNDRY_ENDPOINT = ""
OCR_ENDPOINT = f"{FOUNDRY_ENDPOINT}/providers/mistral/azure/ocr"
#OCR_DEPLOYMENT = "mistral-document-ai-2512"
OCR_DEPLOYMENT = "Mistral-Large-3"

print(f"Foundry resource: {FOUNDRY_ENDPOINT}")
print(f"OCR deployment: {OCR_DEPLOYMENT}")
print(f"OCR endpoint: {OCR_ENDPOINT}")

Foundry resource: 
OCR deployment: Mistral-Large-3
OCR endpoint: /providers/mistral/azure/ocr


## 3. Microsoft Entra ID authentication (recommended)

`DefaultAzureCredential` uses your Azure CLI, Azure PowerShell, or developer-tool sign-in locally. The helper refreshes the access token for each request.

In [4]:
credential = DefaultAzureCredential()
TOKEN_SCOPE = "https://ai.azure.com/.default"


def entra_headers() -> dict[str, str]:
    token = credential.get_token(TOKEN_SCOPE)
    return {
        "Authorization": f"Bearer {token.token}",
        "Content-Type": "application/json",
    }


# Acquire a token now so authentication or RBAC problems fail early.
_ = credential.get_token(TOKEN_SCOPE)
print("Microsoft Entra ID token acquired.")

Microsoft Entra ID token acquired.


## 4. Optional API-key authentication

Your resource currently disables API-key authentication, so this cell is for customer demonstration only. Never place a key directly in the notebook. The Mistral deployment uses the key as a bearer credential; confirm the generated sample on the deployment's **Consume** page if the customer's deployment exposes a different header.

In [5]:
def api_key_headers() -> dict[str, str]:
    api_key = os.getenv("AZURE_MISTRAL_OCR_API_KEY")
    if not api_key:
        raise RuntimeError("Set AZURE_MISTRAL_OCR_API_KEY in .env to use API-key authentication.")
    return {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }


print("API-key helper defined; no key was loaded or displayed.")

API-key helper defined; no key was loaded or displayed.


## 5. Select and encode a document

After adding a PDF, Word document, PowerPoint presentation, or image to `documents`, update `DOCUMENT_PATH` if necessary.

In [ ]:
DOCUMENT_PATH = Path("documents/AdventureWorksPO_HROrder.png")


def document_data_url(path: Path) -> str:
    if not path.is_file():
        raise FileNotFoundError(f"Document not found: {path.resolve()}")

    media_type, _ = mimetypes.guess_type(path.name)
    if media_type is None:
        raise ValueError(f"Could not determine the media type for {path.name}")

    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{media_type};base64,{encoded}"


print(f"Document ready: {DOCUMENT_PATH.is_file()}")

## 6. Send the document to Mistral OCR

This starter request returns page-level Markdown and metadata. It uses Microsoft Entra ID by default; replace `entra_headers()` with `api_key_headers()` to demonstrate key authentication.

In [ ]:
def read_document(path: Path, *, include_images: bool = False) -> dict[str, Any]:
    payload = {
        "model": OCR_DEPLOYMENT,
        "document": {
            "type": "document_url",
            "document_url": document_data_url(path),
        },
        "include_image_base64": include_images,
    }

    response = requests.post(
        OCR_ENDPOINT,
        headers=entra_headers(),
        json=payload,
        timeout=180,
    )
    response.raise_for_status()
    return response.json()


# Run after adding the document and configuring the Foundry resource endpoint.
ocr_result = read_document(DOCUMENT_PATH)
print(json.dumps(ocr_result, indent=2)[:5000])

## 7. Inspect the result

In [ ]:
def summarize_result(result: dict[str, Any]) -> dict[str, Any]:
    pages = result.get("pages", [])
    return {
        "page_count": len(pages),
        "markdown_characters": sum(len(page.get("markdown", "")) for page in pages),
        "image_count": sum(len(page.get("images", [])) for page in pages),
    }


summary = summarize_result(ocr_result)
display(summary)

pages = ocr_result.get("pages", [])
if pages:
    display(Markdown(pages[0].get("markdown", "")))
else:
    print("The OCR response did not contain any pages.")